# CAF Spatial Distribution Analysis

Analyze the spatial distribution of Cancer-Associated Fibroblast (CAF) subtypes
relative to tumor and normal tissue boundaries using grid-based distance mapping.

**Key analyses:**
- NR CAF proportion in peri-tumor vs peri-normal regions across all patients
- Differential gene expression: NR CAF vs Normal Fibroblast
- Reactome pathway enrichment of DEGs
- Spatial visualization of CAF annotations

**Conda environment:** `skny`

## Imports

In [ ]:
import os
import math

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import skny as sk
import stlearn as st
import gseapy as gp
from gseapy import barplot, dotplot
from scipy.stats import wilcoxon

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import seaborn as sns
import cv2
from PIL import Image
import networkx as nx
from adjustText import adjust_text

## Utility Functions

In [ ]:
class MidpointNormalize(colors.Normalize):
    """Normalize the colorbar so that diverging bars work from a prescribed midpoint value.

    Example:
        im = ax1.imshow(array, norm=MidpointNormalize(midpoint=0., vmin=-100, vmax=100))
    """

    def __init__(self, vmin=None, vmax=None, midpoint=None, clip=False):
        self.midpoint = midpoint
        colors.Normalize.__init__(self, vmin, vmax, clip)

    def __call__(self, value, clip=None):
        x, y = [self.vmin, self.midpoint, self.vmax], [0, 0.5, 1]
        return np.ma.masked_array(np.interp(value, x, y), np.isnan(value))

In [ ]:
def cv2pil(image):
    """Convert OpenCV image to PIL Image."""
    new_image = image.copy()
    if new_image.ndim == 2:
        pass
    elif new_image.shape[2] == 3:
        new_image = cv2.cvtColor(new_image, cv2.COLOR_BGR2RGB)
    elif new_image.shape[2] == 4:
        new_image = cv2.cvtColor(new_image, cv2.COLOR_BGRA2RGBA)
    return Image.fromarray(new_image)

In [ ]:
def calculate_distance(grid, pos_marker_ls=None, neg_marker_ls=None, use_label=None):
    """Calculate shortest distance from tumor/normal surface for each grid cell.

    Parameters
    ----------
    grid : AnnData
        Grid object from stlearn.
    pos_marker_ls : list, optional
        Positive marker gene list.
    neg_marker_ls : list, optional
        Negative marker gene list.
    use_label : str, optional
        Column name in grid.obs to use as label.

    Returns
    -------
    AnnData
        Grid with distance annotations added.
    """
    N_ROW = len(grid.uns["grid_yedges"]) - 1
    N_COL = len(grid.uns["grid_xedges"]) - 1

    if use_label is None:
        pos_series = sum([grid.to_df()[i] for i in pos_marker_ls])
        pos_series = pd.Series([1 if i >= 1 else 0 for i in pos_series], index=pos_series.index)

        if neg_marker_ls is not None:
            neg_series = sum([grid.to_df()[i] for i in neg_marker_ls])
            neg_series = pd.Series([1 if i >= 1 else 0 for i in neg_series], index=neg_series.index)
            pos_series = pos_series - neg_series
            pos_series = pd.Series([-1 if i == 0 else i for i in pos_series], index=pos_series.index)
        else:
            pos_series = pd.Series([-1 if i == 0 else i for i in pos_series], index=pos_series.index)

        pos_series.name = "tumor_grid"
        df_grid_tumor = pd.merge(
            pd.DataFrame(index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)]),
            pos_series, right_index=True, left_index=True, how="left"
        ).fillna(-1)
    else:
        pos_series = grid.obs[use_label].copy()
        pos_series.name = "tumor_grid"
        df_grid_tumor = pd.merge(
            pd.DataFrame(index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)]),
            pos_series, right_index=True, left_index=True, how="left"
        ).fillna(-1)

    # Generate image of positive grid
    fig, ax = plt.subplots(figsize=(N_COL, N_ROW), dpi=1, tight_layout=True)
    cmap = matplotlib.cm.viridis
    cmap.set_bad("black", 1.0)
    cmap.set_under(color="black")
    ax.imshow(
        np.array(df_grid_tumor["tumor_grid"]).reshape(N_COL, N_ROW).T,
        cmap=cmap, vmin=0, vmax=1,
    )
    ax.axis("off")
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    plt.close()
    grid.uns["marker"] = img.copy()

    # Median + Gaussian filter
    img_med = cv2.medianBlur(img, ksize=3)
    img_med = cv2.GaussianBlur(img_med, (3, 3), 0)
    grid.uns["marker_median"] = img_med.copy()

    # Delineation (contour detection)
    threshold = 20
    img_gray = cv2.cvtColor(img_med, cv2.COLOR_BGR2GRAY)
    img_blur = img_gray
    ret, img_binary = cv2.threshold(img_blur, threshold, 255, cv2.THRESH_BINARY)
    grid.uns["marker_binary"] = img_binary.copy()
    contours, hierarchy = cv2.findContours(img_binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    contours_ = []
    for i, s in zip(contours, hierarchy[0]):
        contours_ += [i]

    img_binary = cv2.cvtColor(img_binary, cv2.COLOR_GRAY2BGR)
    img_binary_ = np.where(img_binary == (255, 255, 255), (36, 231, 253), (0, 0, 0))
    img_binary_ = np.array(img_binary_, dtype=np.uint8)
    img_color_with_contours = cv2.drawContours(img_binary_, contours_, -1, (0, 255, 0), 1)
    grid.uns["marker_delineation"] = img_color_with_contours

    # Convert image data to graph structure
    image = cv2pil(img_color_with_contours)
    width, height = image.size
    graph = nx.Graph()

    tumor_contour_pixel_ls = []
    tumor_pixel_ls = []
    edge_pixel_ls = []

    for y in range(height):
        for x in range(width):
            pixel_value = image.getpixel((x, y))
            graph.add_node((x, y), color=pixel_value)

            if pixel_value == (0, 255, 0):
                tumor_contour_pixel_ls += [(x, y)]
            elif pixel_value == (253, 231, 36):
                tumor_pixel_ls += [(x, y)]

            if (y == 0) | (y == height - 1) | (x == 0) | (x == width - 1):
                edge_pixel_ls += [(x, y)]

    nodes_ls = list(graph.nodes)

    # Check nodes inside contour
    inside_contour_ls = []
    for node in nodes_ls:
        for i in contours_:
            if cv2.pointPolygonTest(i, node, False) == 1:
                inside_contour_ls += [node]
                break

    tumor_pixel_ls = [i for i in tumor_pixel_ls if i in inside_contour_ls]

    # Create edges between adjacent pixels
    for y in range(height):
        for x in range(width):
            current_node = (x, y)
            neighbors = [(x, y - 1), (x, y + 1), (x - 1, y), (x + 1, y)]
            for neighbor in neighbors:
                if neighbor in graph.nodes:
                    graph.add_edge(current_node, neighbor)

            if (y != height) & (x != width):
                node1 = (x, y)
                node2 = (x + 1, y + 1)
                distance = math.sqrt((node1[0] - node2[0]) ** 2 + (node1[1] - node2[1]) ** 2)
                graph.add_edge(node1, node2, weight=distance)

                node1 = (x + 1, y)
                node2 = (x, y + 1)
                distance = math.sqrt((node1[0] - node2[0]) ** 2 + (node1[1] - node2[1]) ** 2)
                graph.add_edge(node1, node2, weight=distance)

    # Delete contour at the edge of image
    tumor_pixel_ls = list(
        set(tumor_pixel_ls) | (set(tumor_contour_pixel_ls) & set(edge_pixel_ls))
    )
    tumor_contour_pixel_ls = list(
        set(tumor_contour_pixel_ls) - (set(tumor_contour_pixel_ls) & set(edge_pixel_ls))
    )

    # Calculate shortest distance from tumor surface
    shortest_paths = nx.multi_source_dijkstra(graph, tumor_contour_pixel_ls, weight="weight")

    df_shotest = pd.DataFrame.from_dict(shortest_paths[0], orient="index", columns=["euclidean"])
    df_shotest.loc[df_shotest.index[df_shotest.index.isin(tumor_pixel_ls)]] = (
        df_shotest.loc[df_shotest.index[df_shotest.index.isin(tumor_pixel_ls)]] * -1
    )
    df_shotest["euclidean_round"] = df_shotest["euclidean"].round(0)
    df_nodes = pd.DataFrame(index=nodes_ls)

    df_shotest = pd.merge(
        df_nodes, df_shotest, right_index=True, left_index=True, how="left"
    ).fillna(np.nan)

    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close()
    grid.uns["shotest"] = img

    # Delineation visualization
    col_ls = []
    for i in df_shotest["euclidean_round"]:
        if i == 0:
            col_ls += [(0, 255, 0)]
        elif i < 0:
            col_ls += [(0, 255, 255)]
        else:
            col_ls += [(0, 0, 0)]

    col_arr = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)
    grid.uns["marker_median_delineation"] = col_arr.copy()

    # Delineation with 30um interval
    col_ls = []
    for i in df_shotest["euclidean_round"]:
        if i == 0:
            col_ls += [(0, 255, 0)]
        elif (i % 3 == 0) & (i > 0):
            col_ls += [(0, 0, 255)]
        elif (i % 3 == 0) & (i < 0):
            col_ls += [(0, 150, 255)]
        elif i < 0:
            col_ls += [(0, 255, 255)]
        else:
            col_ls += [(0, 0, 0)]

    col_arr = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)
    grid.uns["shotest_30_delineation"] = col_arr.copy()

    # Discretization
    df_shotest["region"] = pd.cut(
        df_shotest.dropna()["euclidean"],
        bins=list(range(-15, 31, 3))
    )

    for i in df_shotest.dropna().region.value_counts()[df_shotest.dropna().region.value_counts() > 100].sort_index().index:
        col_ls = []
        for s in df_shotest["region"]:
            if s == i:
                col_ls += [(255, 255, 255)]
            else:
                col_ls += [(0, 0, 0)]

        col_arr = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)
        grid.uns[f"shotest_region_{i}_delineation"] = col_arr

    setattr(grid, "shortest", df_shotest)

    return grid

## Configuration

In [ ]:
DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

N_CPUS = os.cpu_count()

PATIENTS = ["Pt-1", "Pt-2", "Pt-3", "Pt-4", "Pt-5", "Pt-6", "Pt-7", "Pt-8"]

NR_CAF_CLUSTERS = ["CAF c0", "CAF c3", "CAF c4"]
CR_CAF_CLUSTERS = ["CAF c1"]

## Data Loading

In [ ]:
adata_high_filtered_caf = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_caf.h5ad"))
adata_high_filtered_tumor_normal = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_tumor_normal.h5ad"))

In [ ]:
adata_high_filtered_caf.obs["Cell_type"] = adata_high_filtered_caf.obs["leiden"]
adata_high_filtered_tumor_normal.obs["Cell_type"] = adata_high_filtered_tumor_normal.obs["Cell_type_integrate"]

In [ ]:
adata_high_filtered = ad.concat([adata_high_filtered_caf, adata_high_filtered_tumor_normal])
del adata_high_filtered_caf, adata_high_filtered_tumor_normal

In [ ]:
adata_high_filtered.uns["spatial"] = adata_high_filtered.obsm["spatial"]

## All-Patient NR CAF Proportion Analysis

Compute the proportion of NR CAF in peri-tumor vs peri-normal regions for each patient.
Grid-based spatial analysis at 10 um resolution with distance thresholds.

In [ ]:
tumor_ls = []
normal_ls = []

for patient in PATIENTS:

    adata_high_filtered_ = adata_high_filtered[
        (adata_high_filtered.obs["Patient"] == patient)
        & (adata_high_filtered.obs["Timepoint"] == "Resection")
    ]

    N_COL = int((adata_high_filtered_.obs.imagecol.max() - adata_high_filtered_.obs.imagecol.min()) / 10)
    N_ROW = int((adata_high_filtered_.obs.imagerow.max() - adata_high_filtered_.obs.imagerow.min()) / 10)
    grid = st.tl.cci.grid(
        adata_high_filtered_, n_row=N_ROW, n_col=N_COL,
        n_cpus=N_CPUS, verbose=False, use_label="Cell_type"
    )

    if ("Tumor" in grid.obs["Cell_type"].unique()) & ("Normal" in grid.obs["Cell_type"].unique()):
        grid.obs = pd.merge(
            grid.obs,
            pd.get_dummies(grid.obs["Cell_type"], dtype=int).astype(float).replace(0, -1),
            right_index=True, left_index=True, how="left"
        )

        # Distance from tumor surface
        grid = calculate_distance(grid, use_label="Tumor")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_tumor"]
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        # Distance from normal surface
        grid = calculate_distance(grid, use_label="Normal")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_normal"]
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        grid.obs.loc[grid.obs["Cell_type"].isin(NR_CAF_CLUSTERS), "CAF annotation"] = "NR CAF"
        grid.obs.loc[grid.obs["Cell_type"].isin(CR_CAF_CLUSTERS), "CAF annotation"] = "CR CAF"
        grid.obs["CAF annotation"] = grid.obs["CAF annotation"].fillna("Other")

        df_tumor = grid[
            (grid.obs["euclidean_tumor"] > 0)
            & (grid.obs["euclidean_tumor"] <= 3)
            & (grid.obs["euclidean_normal"] > 3)
        ].obs
        df_normal = grid[
            (grid.obs["euclidean_normal"] > 0)
            & (grid.obs["euclidean_normal"] <= 3)
            & (grid.obs["euclidean_tumor"] > 3)
        ].obs

        tumor_ls += [(df_tumor["CAF annotation"].value_counts() / len(df_tumor))["NR CAF"]]
        normal_ls += [(df_normal["CAF annotation"].value_counts() / len(df_normal))["NR CAF"]]

    else:
        grid.obs = pd.merge(
            grid.obs,
            pd.get_dummies(grid.obs["Cell_type"], dtype=int).astype(float).replace(0, -1),
            right_index=True, left_index=True, how="left"
        )

        grid = calculate_distance(grid, use_label="Tumor")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_tumor"]
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        grid.obs.loc[grid.obs["Cell_type"].isin(NR_CAF_CLUSTERS), "CAF annotation"] = "NR CAF"
        grid.obs.loc[grid.obs["Cell_type"].isin(CR_CAF_CLUSTERS), "CAF annotation"] = "CR CAF"
        grid.obs["CAF annotation"] = grid.obs["CAF annotation"].fillna("Other")

        df_tumor = grid[(grid.obs["euclidean_tumor"] > 0) & (grid.obs["euclidean_tumor"] <= 3)].obs
        tumor_ls += [(df_tumor["CAF annotation"].value_counts() / len(df_tumor))["NR CAF"]]
        normal_ls += [np.nan]

In [ ]:
df = pd.DataFrame(
    [normal_ls, tumor_ls],
    columns=PATIENTS,
    index=["Normal", "Tumor"]
).T

In [ ]:
pd.melt(df)

## Statistical Test & Visualization

Wilcoxon signed-rank test for paired comparison of NR CAF proportions
between peri-tumor and peri-normal regions.

In [ ]:
stat, p_value = wilcoxon(df["Normal"], df["Tumor"])

df_reset = df.reset_index()

plt.figure(figsize=(4, 2.5))

sns.violinplot(
    x="variable", y="value", hue="variable", data=pd.melt(df),
    inner="quartile", palette=["blue", "red"], legend=False, alpha=0.7
)

for i in range(len(df_reset)):
    plt.plot(
        ["Normal", "Tumor"],
        [df_reset.loc[i, "Normal"], df_reset.loc[i, "Tumor"]],
        marker="o", color="black", alpha=0.7
    )

plt.text(
    0.5, max(pd.melt(df)["value"]) + 0.13, f"p = {p_value:.4f}",
    ha="center", fontsize=13, color="black"
)

plt.xlabel("")
plt.ylabel("Density (/10\u03bcm\u00b2)", fontsize=12)
plt.title("NR CAF in peri-tumor/normal", fontsize=12)
plt.xticks(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "NR_CAF_tumor_normal.png"), dpi=300)
plt.show()

## DEG Analysis: NR CAF vs Normal Fibroblast

Build a combined grid across all patients, then run Wilcoxon rank-sum test
to identify differentially expressed genes between NR CAF and peri-normal fibroblasts.

In [ ]:
flag = 0
for patient in PATIENTS:

    adata_high_filtered_ = adata_high_filtered[
        (adata_high_filtered.obs["Patient"] == patient)
        & (adata_high_filtered.obs["Timepoint"] == "Resection")
    ]

    N_COL = int((adata_high_filtered_.obs.imagecol.max() - adata_high_filtered_.obs.imagecol.min()) / 10)
    N_ROW = int((adata_high_filtered_.obs.imagerow.max() - adata_high_filtered_.obs.imagerow.min()) / 10)
    grid = st.tl.cci.grid(
        adata_high_filtered_, n_row=N_ROW, n_col=N_COL,
        n_cpus=N_CPUS, verbose=False, use_label="Cell_type"
    )

    if ("Tumor" in grid.obs["Cell_type"].unique()) & ("Normal" in grid.obs["Cell_type"].unique()):
        grid.obs = pd.merge(
            grid.obs,
            pd.get_dummies(grid.obs["Cell_type"], dtype=int).astype(float).replace(0, -1),
            right_index=True, left_index=True, how="left"
        )

        grid = calculate_distance(grid, use_label="Tumor")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_tumor"]
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        grid = calculate_distance(grid, use_label="Normal")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_normal"]
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        grid.obs.loc[grid.obs["Cell_type"].isin(NR_CAF_CLUSTERS), "CAF annotation"] = "NR CAF"
        grid.obs.loc[grid.obs["Cell_type"].isin(CR_CAF_CLUSTERS), "CAF annotation"] = "CR CAF"
        grid.obs["CAF annotation"] = grid.obs["CAF annotation"].fillna("Other")
        grid.obs["Patient"] = patient

        if flag == 0:
            grid_res = grid
            flag += 1
        else:
            grid_res = ad.concat([grid_res, grid])

In [ ]:
grid_res.obs.loc[
    (grid_res.obs["Cell_type"].str.contains("CAF"))
    & (grid_res.obs["euclidean_normal"] <= 3)
    & (grid_res.obs["euclidean_normal"] > 0)
    & (grid_res.obs["euclidean_tumor"] > 3),
    "CAF_annotation_"
] = "Fibroblast"

grid_res.obs.loc[
    grid_res.obs["Cell_type"].isin(NR_CAF_CLUSTERS), "CAF_annotation_"
] = "NR CAF" 

In [ ]:
del grid

group_name = "NR CAF"
sc.tl.rank_genes_groups(grid_res, "CAF_annotation_", groups=[group_name], reference="Fibroblast", method="wilcoxon")
degs = sc.get.rank_genes_groups_df(grid_res, group=group_name)

In [ ]:
pval_threshold = 1e-10
logfc_threshold = 1

degs["color"] = "grey"
degs.loc[
    (degs["pvals_adj"] < pval_threshold) & (degs["logfoldchanges"] > logfc_threshold), "color"
] = "red"
degs.loc[
    (degs["pvals_adj"] < pval_threshold) & (degs["logfoldchanges"] < -logfc_threshold), "color"
] = "blue"
degs["pvals_adj"] = degs["pvals_adj"].replace(0, 1e-300)

### Volcano Plot

In [ ]:
plt.figure(figsize=(6, 5))

for color, group in degs.groupby("color"):
    plt.scatter(group["logfoldchanges"], -np.log10(group["pvals_adj"]), color=color, alpha=0.7, label=color)

plt.xlim(-7, 7)

top_genes_up = degs[degs["logfoldchanges"] > logfc_threshold].sort_values(by="pvals_adj").head(20)
top_genes_down = degs[degs["logfoldchanges"] < -logfc_threshold].sort_values(by="pvals_adj").head(20)

texts = []
for i, row in top_genes_up.iterrows():
    if row["color"] == "red":
        texts.append(plt.text(row["logfoldchanges"], -np.log10(row["pvals_adj"]), row["names"], fontsize=11))
for i, row in top_genes_down.iterrows():
    if row["color"] == "blue":
        texts.append(plt.text(row["logfoldchanges"], -np.log10(row["pvals_adj"]), row["names"], fontsize=11))

adjust_text(
    texts,
    force_text=100, force_points=100, expand_text=(1.2, 1.5),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1.0)
)

plt.xlabel("Log2 Fold Change", fontsize=16)
plt.ylabel("-Log10 Adjusted p-value", fontsize=16)
plt.grid(False)
plt.legend(["Fib", "Neutral", group_name], loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=12)
plt.tight_layout()
plt.show()

## Pathway Enrichment (Reactome)

Run Reactome pathway enrichment on upregulated and downregulated DEGs
using gseapy/Enrichr.

### Upregulated genes (NR CAF)

In [ ]:
deg_ls = degs[degs["color"] == "red"]["names"].tolist()

enr = gp.enrichr(
    gene_list=deg_ls,
    gene_sets=["Reactome_Pathways_2024"],
    organism="human",
    outdir=None,
)
df_temp = enr.results.copy()

In [ ]:
barplot(enr.res2d, title="Reactome", color="r", top_term=15)

In [ ]:
df_enr = enr.results
df_enr["group"] = "NR CAF (vs Fibroblast)"
df_enr.to_csv(os.path.join(RESULTS_DIR, "reactome_NR_CAF_vs_Fibroblast.tsv"), sep="\t")

### Downregulated genes (Fibroblast)

In [ ]:
deg_ls = degs[degs["color"] == "blue"]["names"].tolist()

enr = gp.enrichr(
    gene_list=deg_ls,
    gene_sets=["Reactome_Pathways_2024"],
    organism="human",
    outdir=None,
)
df_temp = enr.results.copy()

In [ ]:
barplot(enr.res2d, title="Reactome", color="b", top_term=15)

In [ ]:
for gene in degs[degs["color"] == "red"]["names"]:
    print(gene)

## Spatial Visualization

Annotate and visualize CAF subtypes spatially across patients.

In [ ]:
grid_res.obs.loc[
    (grid_res.obs["Cell_type"].str.contains("CAF"))
    & (grid_res.obs["euclidean_normal"] <= 3)
    & (grid_res.obs["euclidean_normal"] > 0)
    & (grid_res.obs["euclidean_tumor"] > 3),
    "CAF_annotation_"
] = "Fibroblast"

grid_res.obs.loc[
    grid_res.obs["Cell_type"].isin(NR_CAF_CLUSTERS), "CAF_annotation_"
] = "NR CAF" 

In [ ]:
grid_res.obs.loc[grid_res.obs["euclidean_normal"] <= 0, "CAF_annotation_"] = "Normal"
grid_res.obs.loc[grid_res.obs["euclidean_tumor"] <= 0, "CAF_annotation_"] = "Tumor" 

In [ ]:
grid_res.obs["CAF_annotation_"] = pd.Categorical(
    grid_res.obs["CAF_annotation_"],
    categories=["Tumor", "Normal", "NR CAF", "Fibroblast"]
)

In [ ]:
grid_res_normal = grid_res

In [ ]:
grid_res_normal = grid_res_normal[
    grid_res_normal.obs.index.isin(grid_res_normal.obs["CAF_annotation_"].dropna().index)
]

In [ ]:
grid_res_normal.obs.index = grid_res_normal.obs.index + "_" + grid_res_normal.obs["Patient"]

In [ ]:
grid_res_normal = grid_res_normal[grid_res_normal.obs["CAF_annotation_"].sort_values().index, :]

In [ ]:
grid_res_normal = grid_res_normal[~grid_res_normal.obs["CAF_annotation_"].isna()]

In [ ]:
del grid_res

In [ ]:
grid_res_normal.obs["CAF_annotation"] = grid_res_normal.obs["CAF_annotation_"].astype(str)

In [ ]:
grid_res_normal.obs.loc[
    grid_res_normal.obs["CAF_annotation"] == "SD CAF", "CAF_annotation"
] = "NR CAF" 

In [ ]:
grid_res_normal.obs["CAF_annotation"] = pd.Categorical(
    grid_res_normal.obs["CAF_annotation"],
    categories=["Tumor", "Normal", "NR CAF", "Fibroblast"]
)

In [ ]:
sq.pl.spatial_scatter(
    grid_res_normal[grid_res_normal.obs["Patient"] == "Pt-7"],
    library_id="spatial",
    shape=None,
    color=["CAF_annotation"],
    wspace=0.4,
    palette=matplotlib.colors.ListedColormap(["black", "gray", "#F9B796", "#D6FFD0"]),
)